<a href="https://colab.research.google.com/github/Priyanshu-223/Fine-tuning-open-source-llm-using-PFT/blob/main/LLM_Fine_Tuning_PEFT_LoRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets peft accelerate

In [ ]:
import torch

print("GPU Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU Available: True
GPU: Tesla T4


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float32
)

device = "cuda" if torch.cuda.is_available() else "cpu"

model = model.to(device)

print("Model loaded successfully")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded successfully


In [ ]:
def generate_response(current_model, question):

    messages = [
        {"role": "user", "content": question}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(device)

    current_model.eval()

    with torch.no_grad():
        outputs = current_model.generate(
            **inputs,
            max_new_tokens=500,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    )

In [ ]:
question = "What is a variable in Python?"

original_response = generate_response(model, question)

print("Original Model Response:")
print(original_response)

Original Model Response:
In Python, a variable is an identifier that stores data and can be used to refer to the same value later on. It is a fundamental concept of programming languages and plays a crucial role in defining variables and their scope.

There are two types of variables in Python: global variables and local variables. Global variables are declared outside any function or block and can be accessed from anywhere within the program. Local variables are declared inside functions and cannot be accessed from other parts of the program.

Variables are used to store values for later use. They can hold different data types such as integers, strings, floats, etc. The type of a variable determines its storage location in memory.

Python also supports keyword arguments, which allow you to pass additional arguments to a function without having to specify them explicitly. This feature is useful when you want to customize the behavior of a function based on certain conditions.

Overall,

In [ ]:
from datasets import Dataset

data = {
    "question": [
        "What is Python?",
        "What is a variable?",
        "What is a list?",
        "What is a tuple?",
        "What is a dictionary?",
        "What is a function?",
        "What is a loop?",
        "What is a string?",
        "What is an integer?",
        "What is a conditional statement?"
    ],

    "answer": [
        "Python is a high-level programming language.",
        "A variable is a named storage location used to store a value.",
        "A list is a collection that stores multiple elements and is mutable.",
        "A tuple is an ordered collection that cannot be modified after creation.",
        "A dictionary stores data in key-value pairs.",
        "A function is a reusable block of code that performs a specific task.",
        "A loop repeats a block of code multiple times.",
        "A string is a sequence of characters.",
        "An integer is a whole number without a decimal part.",
        "A conditional statement executes code based on whether a condition is true or false."
    ]
}

dataset = Dataset.from_dict(data)

print(dataset)

Dataset({
    features: ['question', 'answer'],
    num_rows: 10
})


In [ ]:
def tokenize_data(batch):

    texts = []

    for question, answer in zip(batch["question"], batch["answer"]):

        messages = [
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer}
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False
        )

        texts.append(text)

    return tokenizer(
        texts,
        truncation=True,
        max_length=256
    )


tokenized_dataset = dataset.map(
    tokenize_data,
    batched=True,
    remove_columns=["question", "answer"]
)

print(tokenized_dataset[0])

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

{'input_ids': [151644, 8948, 198, 2610, 525, 1207, 16948, 11, 3465, 553, 54364, 14817, 13, 1446, 525, 264, 10950, 17847, 13, 151645, 198, 151644, 872, 198, 3838, 374, 13027, 30, 151645, 198, 151644, 77091, 198, 30280, 374, 264, 1550, 11591, 15473, 4128, 13, 151645, 198], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [ ]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"]
)

In [ ]:
model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./qwen-lora-results",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    learning_rate=2e-4,
    logging_steps=1,
    fp16=torch.cuda.is_available(),
    save_strategy="no",
    report_to="none"
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
    processing_class=tokenizer
)

In [ ]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 151643}.


Step,Training Loss
1,4.324273
2,4.113937
3,4.117782
4,3.731254
5,3.536262
6,3.419262
7,3.079074
8,3.025892
9,2.957770
10,3.001625


TrainOutput(global_step=25, training_loss=2.716983981132507, metrics={'train_runtime': 3.6498, 'train_samples_per_second': 13.699, 'train_steps_per_second': 6.85, 'total_flos': 5277652655616.0, 'train_loss': 2.716983981132507, 'epoch': 5.0})

In [ ]:
model.save_pretrained("./qwen-lora-adapter")

tokenizer.save_pretrained("./qwen-lora-adapter")

print("Fine-tuned adapter saved successfully")

Fine-tuned adapter saved successfully


In [ ]:
model.config.use_cache = True

fine_tuned_response = generate_response(
    model,
    question
)

print("Fine-tuned Model Response:")
print(fine_tuned_response)

Fine-tuned Model Response:
In Python, a variable is an identifier that stores data and can be used to refer to the same value later on. It is a fundamental concept of programming languages and plays a crucial role in defining variables and their scope.

There are two types of variables in Python: global variables and local variables. Global variables are declared outside any function or block and can be accessed from anywhere within the program. Local variables are declared inside functions and cannot be accessed from other parts of the program.

Variables are used to store values for later use. They can hold different data types such as integers, strings, floats, etc. The type of a variable determines its storage location in memory.

Python also supports keyword arguments, which allow you to pass additional arguments to a function without having to specify them explicitly. This feature is useful when you want to customize the behavior of a function based on certain conditions.

Overal

In [ ]:
print("Before Fine-tuning:")
print(original_response)

print("\nAfter Fine-tuning:")
print(fine_tuned_response)

Before Fine-tuning:
In Python, a variable is an identifier that stores data and can be used to refer to the same value later on. It is a fundamental concept of programming languages and plays a crucial role in defining variables and their scope.

There are two types of variables in Python: global variables and local variables. Global variables are declared outside any function or block and can be accessed from anywhere within the program. Local variables are declared inside functions and cannot be accessed from other parts of the program.

Variables are used to store values for later use. They can hold different data types such as integers, strings, floats, etc. The type of a variable determines its storage location in memory.

Python also supports keyword arguments, which allow you to pass additional arguments to a function without having to specify them explicitly. This feature is useful when you want to customize the behavior of a function based on certain conditions.

Overall, vari

In [ ]:
original_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float32
).to(device)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [ ]:
test_question = "Explain the difference between a list and a tuple in Python."

print("Original Model:")
print(generate_response(original_model, test_question))

print("\nFine-tuned Model:")
print(generate_response(model, test_question))

Original Model:
In Python, both lists and tuples are used to store collections of items, but they have some key differences:

1. **List**:
   - A list is an ordered collection of elements that can be accessed using an index.
   - Lists are mutable (you can change their contents after creation).
   - Lists are defined with square brackets `[]`.
   - Lists can contain different data types.

2. **Tuple**:
   - A tuple is also an ordered collection of elements that can be accessed using an index.
   - Tuples are immutable (you cannot change their contents after creation).
   - Tuples are defined with parentheses `()`.
   - Tuples are similar to lists, but they do not allow duplicates or modification of existing elements.

Here's an example to illustrate these differences:

```python
# List Example
my_list = [1, 2, 3]
print(my_list)  # Output: [1, 2, 3]

# Tuple Example
my_tuple = (1, 2, 3)
print(my_tuple)  # Output: (1, 2, 3)

# Accessing Elements
print(my_list[0])  # Output: 1
print(my_tu